# Peng Robinsons EOS's calculator 
This notebook uses the Peng Robinson's equation of state (PR-EoS) to calculate the volume of fluid, provided the temperature, pressure, and critical properties. The PR-EoS is given below:

$$ P = \frac{RT}{v_m-b_m} - \frac{a_m}{v_m(v_m+b_m)+b_m(v_m-b_m)} $$

Where:

$$ a_m = \sum_i \sum_j {y_i y_j (1 - K_{ij})\sqrt{a_i a_j}} $$
$$ b_m = \sum{y_i b_i} $$

The notebook uses a non-linear root finding algorithm provided by `scipy.optimize.fsolve` to determine the volume. Whether the converged volume is the liquid- or vapour-phase volume therefore depends on the initial guess.

## Import libraries

In [13]:
from scipy.integrate import solve_ivp
from scipy.integrate import quad
from scipy.optimize import fsolve
import numpy as np

## Parameters
The code requires the following parameters:

| Mathematical symbol | Variable name | Description |
| --- | --- | --- |
| $R$ | `R` | Universal gas constant (J/(mol K)). |
| $T_{C,i}$ | `Tc` | Critical temperature of each component (K). |
| $P_{C,i}$ | `Pc` | Critical pressure of each component (Pa). |
| $\omega_i$ | `omega` | Acentric factor of each component. |
| $\Delta_f H_i^\circ$ | `hf` | Enthalpy of formation of each component (J/mol). |
| $A_i, B_i, C_i, D_i, E_i$ | `cP_constants` | Coefficients for each component's ideal-gas heat-capacity correlation. |
| $K_{ij}$ | `K` | Binary interaction parameter for component pair $i,j$. |
| $T_{\mathrm{ref}}$ | `T_ref` | Reference temperature for the ideal-gas enthalpy change (K). |

Using these parameters, it can calculate the molar volume and mixture enthalpy for any vapour mixture using the PR-EoS, provided the temperature, pressure and molar composition.

In [14]:
#Constants:

R = 8.314 #Universal gas constant in J/(mol*K)

Tc = {'H2': 33.145, 'N2': 126.192, 'NH3': 405.56} #Critical temperatures in K
Pc = {'H2': 1.2964e6, 'N2': 3.3958e6, 'NH3': 11.3634e6} #Critical pressures in Pa  #TML: use 1.2964e6 instead of 1.2964 * 10**6
omega = {'H2': -0.219, 'N2': 0.0372, 'NH3': 0.256} #Acentric factors
hf = {'H2': 0, 'N2': 0, 'NH3': -45.90 * 1000} #Enthalpy of formation in J/mol

#Ideal heat capacities constants
cP_constants = {'H2': {'A': 33.066178, 'B': -11.363417, 'C': 11.432816, 'D': -2.772874, 'E': -0.158558},
                'N2': {'A': 19.50583, 'B': 19.88705, 'C': -8.598535, 'D': 1.369784, 'E': 0.527601},
                'NH3': {'A': 19.99563, 'B': 49.77119, 'C': -15.37599, 'D': 1.921168, 'E': 0.189174}}

# Binary interaction parameters for Peng-Robinson EOS:
# TML: you can also define an empty dictionary and then add entries to it, e.g., dict = {}; dict['key1'] = value1; dict['key2'] = value2; ...
K = {}
K['H2'] = {'H2': 0,   #TML: must include own entry
           'N2': 0,   #Binary interaction parameter for H2-N2 - currently using placeholders as I do not have the value yet
           'NH3':0}   #Binary interaction parameter for H2-NH3 - currently using placeholders as I do not have the value yet
K['N2'] = {'H2': 0,   #Binary interaction parameter for N2-H2 - currently using placeholders as I do not have the value yet   # TML: must repeat symmetric entries
           'N2': 0,   #TML: must include own entry
           'NH3': 0}  #Binary interaction parameter for N2-NH3 - currently using placeholders as I do not have the value yet
K['NH3'] = {'H2': 0,  #Binary interaction parameter for NH3-H2 - currently using placeholders as I do not have the value yet  # TML: must repeat symmetric entries
            'N2': 0,  #Binary interaction parameter for NH3-N2 - currently using placeholders as I do not have the value yet  # TML: must repeat symmetric entries
            'NH3':0}  #TML: must include own entry  

#System parameters of mixture at feed:
T_ref = 298.15 #Reference temperature in K
T = 700 
P = 2 * 10**6
y = {'H2': 1/4, 'N2': 3/4, 'NH3': 0} #Mole fractions of components in the mixture

The function below is used to calculate the component parameters $\alpha_i$, $\kappa_i$, $a_i$, $b_i$ and $a_c$ using the following equations:

$$ \kappa_i = 0.37464 + 1.54226 \omega_i - 0.26992 \omega_i^2 $$
$$ \alpha_i = [1 + \kappa_i (1-\sqrt{\frac{T}{T_{c,i}}})]^2 $$
$$ a_{c,i} =  0.45724  \frac{R^2 T_{c,i}^2}{P_c} $$
$$ a_i = a_{c,i} \alpha_i $$
$$ b_i = 0.07779 \frac{R T_c}{P_c} $$



In [15]:
def component_parameters_calculation(T, Tc, Pc, omega):
    a_c = 0.45724 * R**2 * Tc**2 / Pc
    kappa = 0.37464 + 1.54226 * omega - 0.26992 * omega**2
    alpha = (1 + kappa * (1 - (T / Tc)**0.5))**2
    a = a_c * alpha
    b = 0.07779 * R * Tc / Pc
    return a, b, a_c, kappa

def component_parameters(T,y):
    a, b, a_c, kappa = {}, {}, {}, {}
    for component in y:
        a[component], b[component],a_c[component],kappa[component] = component_parameters_calculation(T, Tc[component], Pc[component], omega[component])
    return a, b, a_c, kappa

a, b, a_c, kappa = component_parameters(T,y)

The function below is used to calculate the mixture parameters $a_m$ and $b_m$ using the following equations:

$$ a_m = \sum_i \sum_j {y_i y_j (1 - K_{ij})\sqrt{a_i a_j}} $$
$$ b_m = \sum{y_i b_i} $$

In [16]:
# TML: we can write the individual terms in the summation. It looks more like the original mathematical expression, which is referenced directly above.
def mixture_parameters(y, a, b, K):
    a_m, b_m = 0, 0
    for component_1 in y:
        b_m += y[component_1]*b[component_1]
        for component_2 in y:
            a_m += y[component_1]*y[component_2] * (1- K[component_1][component_2]) * (a[component_1] * a[component_2])**0.5
    return a_m, b_m

a_m, b_m = mixture_parameters(y, a, b, K)

The function below uses the Peng-Robinson EoS to calculate the volume and pressure of the fluid using the following equation:

$$ P = \frac{RT}{v_m-b_m} - \frac{a_m}{v_m(v_m+b_m)+b_m(v_m-b_m)} $$

In [18]:
def volume (T, P, y, a_m,b_m):
     v_m0 = R * T / P #Initial guess for molar volume in m^3/mol
     volume = lambda v_m: (P - R*T / (v_m - b_m) + a_m / (v_m * (v_m + b_m) + b_m * (v_m - b_m)))  #TML: better to define one line function as lambda function
     v_m = fsolve(volume, (v_m0))[0] #Molar volume of the mixture in m^3/mol
     return(v_m)

vmix = volume (T, P, y, a_m,b_m)
print(f"Volume of the gas mixture: {vmix:.8f} m^3/mol")

def pressure (T, v_m, a_m,b_m):
    P = R*T / (v_m - b_m) - a_m / (v_m * (v_m + b_m) + b_m * (v_m - b_m)) #Pressure in Pa
    return P

Volume of the gas mixture: 0.00292793 m^3/mol


The function below is used to calculate the ideal gas enthalpy change and compressibility factor of the mixture using the following equations:

$$ c_{p,i} (T)= A_i + B_i t +C_i t^2 + D_i t^3 + \frac {E_i}{t^2} $$
where, $$ t = \frac{T}{1000}$$
and,
$$ h_m^{ig}= ∑c_i (Δh_{f,i}^o+ \int_{T^o}^{T} c_{p,i}^{ig}(T) dT) $$
$$ Z = \frac {Pv_m}{RT}$$
$$ B = \frac{b_mP}{RT}





In [19]:
Z = P * vmix / (R * T)
B = b_m * P / (R * T)

def ig_enthalpies (T, vmix, y, hf):
    integrand_cP = lambda T, A, B, C, D, E: A + B * T / 1000 + C * (T / 1000)**2 + D * (T / 1000)**3 + E / (T / 1000)**2
    c, dh_ig_m = 1/vmix, 0
    for component in y:
        dh_ig_m += y[component] * (hf[component] + quad(integrand_cP, T_ref, T, args=(cP_constants[component]['A'] , cP_constants[component]['B'] , cP_constants[component]['C'], cP_constants[component]['D'], cP_constants[component]['E']))[0]) #cp_H2 in J/mol
        # c += y[component] * P / (Z * R * T)
        
    dh_ig_m = c * (dh_ig_m)
    return (c, dh_ig_m)
    
c, dh_ig_m = ig_enthalpies (T, vmix, y, hf)

The function below is used to calculate the residual enthalpies for each of the components and ultimately, the enthalpy of the mixture using the following equations:

$$ \frac{d\alpha_i}{dT} = - \frac{\kappa_i (1 + \kappa_i (1-\sqrt{\frac{T}{T_{c,i}}}))}{\sqrt{TT_{c,i}}} $$

$$ \frac{da_i}{dT} = a_{c,i} \frac{d\alpha_i}{dT} $$

$$ \frac {da_m}{dT} = \sum_i \sum_j {y_i y_j (1 - K_{ij})} \frac{1}{2} (\sqrt{\frac{a_j}{a_i}} \frac{da_i}{dT} + \sqrt{\frac{a_i}{a_j}}\frac{da_j}{dT}) $$

$$ h_m^R = RT(Z - 1) + \frac{T\frac{da_m}{dT}-a_m}{(2√2 b_m )}  
ln⁡(\frac{Z+(1+√2)B}{Z+(1-√2)B})

In [21]:
def enthalpy (T, y, a, kappa, Tc, a_c, a_m, b_m, K, c, dh_ig_m):    
    dalpha_dT = lambda T, kappa, Tc: -kappa * (1 + kappa * (1 - (T / Tc)**0.5)) / ((T * Tc)**0.5)
    da_dT = {}
    da_m = 0
    for component_1 in y:
        da_dT[component_1] = a_c[component_1] * dalpha_dT(T, kappa[component_1], Tc[component_1])

    for component_1 in y:
            for component_2 in y:
                da_m += y[component_1]*y[component_2]*(1 - K[component_1][component_2]) * 0.5 \
                * ((a[component_2]/a[component_1])**0.5 * da_dT[component_1] + (a[component_1]/a[component_2])**0.5 * da_dT[component_2])

    h_residual = c * (R * T * (Z - 1) + (T * da_m - a_m) / (b_m * 2 * 2**0.5) * np.log((Z + B * (1 + 2**0.5)) / (Z + B * (1 - 2**0.5))))
    h_m = dh_ig_m + h_residual
    return h_m

hmix = enthalpy(T, y, a, kappa, Tc, a_c, a_m, b_m, K, c, dh_ig_m)

print(f"Enthalpy of the gas mixture: {hmix:.2f} J/m^3, which is kinda equal to {hmix:.3e} J/m^3")


Enthalpy of the gas mixture: 4084879.29 J/m^3, which is kinda equal to 4.085e+06 J/m^3


## Final Call Function

The function below combines all the calculations done in this notebook into a single call. It takes an input of temperature, pressure and molar composition of the mixture and returns the molar volume, total molar concentration and enthalpy of the mixture per unit volume of gas. 

In [22]:
def properties (T, P, y):
    a, b, a_c, kappa = component_parameters(T,y)
    amix, bmix = mixture_parameters(y, a, b, K)
    vmix = volume (T, P, y, amix, bmix)
    Z = P * vmix / (R * T)
    B = bmix * P / (R * T)
    c, dh_ig_m = ig_enthalpies (T, vmix, y, hf)
    hmix = enthalpy(T, y, a, kappa, Tc, a_c, amix, bmix, K, c, dh_ig_m)
    return vmix, hmix, c